In [ ]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [ ]:
df = pd.read_excel('stat test_01APR25.xlsx')
df['sex'] = df['sex'].apply(lambda x: 0 if x == 'MALE' else 1)

In [ ]:
exclude_cols = ['Asthma', 'donor', 'sex']
df.loc[:, ~df.columns.isin(exclude_cols)] = scaler.fit_transform(df.loc[:, ~df.columns.isin(exclude_cols)])

In [ ]:
control = df[(df['Asthma'] == 'Control')]
history = df[(df['Asthma'] == 'History')]
fatal = df[(df['Asthma'] == 'Fatal')]

In [ ]:
control.loc[:, ~control.columns.isin(exclude_cols)] = scaler.fit_transform(control.loc[:, ~control.columns.isin(exclude_cols)])
history.loc[:, ~history.columns.isin(exclude_cols)] = scaler.fit_transform(history.loc[:, ~history.columns.isin(exclude_cols)])
fatal.loc[:, ~fatal.columns.isin(exclude_cols)] = scaler.fit_transform(fatal.loc[:, ~fatal.columns.isin(exclude_cols)])

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.9)

X = df.drop(['Asthma', 'donor', 'sex'], axis=1)
X = pca.fit_transform(X)

pca_columns = [f'PC{i+1}' for i in range(X.shape[1])]
X_df= pd.DataFrame(X, columns=pca_columns)

X_df = pd.concat([X_df, df[['Asthma', 'donor','sex']]], axis=1)
X_df['sex'] = X_df['sex'].apply(lambda x: 'Male' if x == 0 else 'Female')

In [ ]:
palette = sns.color_palette('Set1', n_colors=X_df['Asthma'].nunique())
sex_categories = X_df['Asthma'].unique()
color_dict = dict(zip(sex_categories, palette))

# Create scatter plot with mapped colors
scatter = sns.scatterplot(x='PC1', y='PC2', data=X_df, hue='Asthma', palette=color_dict)

# Annotate each point with its 'donor' value
for i, row in X_df.iterrows():
    plt.text(row['PC1'], row['PC2'], str(row['donor']), fontsize=8, ha='right', alpha=0.7)

# Compute centroids and plot circles
all_x, all_y = [], []  # Store all points for axis limit adjustment
for sex_group in sex_categories:
    subset = X_df[X_df['Asthma'] == sex_group]
    if not subset.empty:
        # Compute centroid
        centroid_x, centroid_y = subset[['PC1', 'PC2']].mean()
        
        # Compute max Euclidean distance for circle radius
        max_distance = np.max(np.sqrt((subset['PC1'] - centroid_x) ** 2 + (subset['PC2'] - centroid_y) ** 2))
        
        # Draw a circle with the same color as the points
        circle = plt.Circle((centroid_x, centroid_y), max_distance, color=color_dict[sex_group], fill=False, linestyle='--', linewidth=2)
        plt.gca().add_patch(circle)

        # Store values to adjust axis limits
        all_x.extend(subset['PC1'])
        all_y.extend(subset['PC2'])
        all_x.append(centroid_x + max_distance)
        all_x.append(centroid_x - max_distance)
        all_y.append(centroid_y + max_distance)
        all_y.append(centroid_y - max_distance)

# Labels and title
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA')

# Expand axis limits to fit the circles
plt.xlim(min(all_x) - 2, max(all_x) + 2)
plt.ylim(min(all_y) - 2, max(all_y) + 2)

# Adjust legend positioning
plt.legend(title='Asthma', bbox_to_anchor=(1.05, 1), loc='upper left')

# Final layout adjustment
plt.tight_layout()
plt.show()

In [ ]:
palette = sns.color_palette('Set1', n_colors=X_df['sex'].nunique())
sex_categories = X_df['sex'].unique()
color_dict = dict(zip(sex_categories, palette))

# Create scatter plot with mapped colors
scatter = sns.scatterplot(x='PC1', y='PC2', data=X_df, hue='sex', palette=color_dict)

# Annotate each point with its 'donor' value
for i, row in X_df.iterrows():
    plt.text(row['PC1'], row['PC2'], str(row['donor']), fontsize=8, ha='right', alpha=0.7)

# Compute centroids and plot circles
all_x, all_y = [], []  # Store all points for axis limit adjustment
for sex_group in sex_categories:
    subset = X_df[X_df['sex'] == sex_group]
    if not subset.empty:
        # Compute centroid
        centroid_x, centroid_y = subset[['PC1', 'PC2']].mean()
        
        # Compute max Euclidean distance for circle radius
        max_distance = np.max(np.sqrt((subset['PC1'] - centroid_x) ** 2 + (subset['PC2'] - centroid_y) ** 2))
        
        # Draw a circle with the same color as the points
        circle = plt.Circle((centroid_x, centroid_y), max_distance, color=color_dict[sex_group], fill=False, linestyle='--', linewidth=2)
        plt.gca().add_patch(circle)

        # Store values to adjust axis limits
        all_x.extend(subset['PC1'])
        all_y.extend(subset['PC2'])
        all_x.append(centroid_x + max_distance)
        all_x.append(centroid_x - max_distance)
        all_y.append(centroid_y + max_distance)
        all_y.append(centroid_y - max_distance)

# Labels and title
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA')

# Expand axis limits to fit the circles
plt.xlim(min(all_x) - 2, max(all_x) + 2)
plt.ylim(min(all_y) - 2, max(all_y) + 2)

# Adjust legend positioning
plt.legend(title='sex', bbox_to_anchor=(1.05, 1), loc='upper left')

# Final layout adjustment
plt.tight_layout()
plt.show()

In [ ]:
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

plt.figure(figsize=(10, 6))
plt.plot(np.arange(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', linestyle='-', color='r', label='Cumulative Explained Variance')
plt.axhline(y=0.9, color='gray', linestyle='--', label='90% Variance Explained')  # 90% threshold line
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("Cumulative Variance Explained by PCA")
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
features = df.drop(['Asthma', 'donor', 'sex'], axis=1).columns
pca_loadings = pd.DataFrame(pca.components_, columns=features, index=[f'PC{i+1}' for i in range(pca.n_components_)])
top_5_features = pca_loadings.abs().apply(lambda x: x.nlargest(5).index.tolist(), axis=1)
print(top_5_features)

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(X[:, 0], X[:, 1], c=df['IL-33'], cmap='viridis')
plt.colorbar(label='IL-33')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA')
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(X[:, 0], X[:, 1], c=df['TNFa'], cmap='viridis')
plt.colorbar(label='TNFa')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA')
plt.show()

In [ ]:
# Extract PC1 and PC2 loadings
pc1_loadings = pca_loadings.loc["PC1"].sort_values(key=abs, ascending=False)
pc2_loadings = pca_loadings.loc["PC2"].sort_values(key=abs, ascending=False)

# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# PC1 plot
sns.barplot(x=pc1_loadings.values[:10], y=pc1_loadings.index[:10], ax=axes[0], color='blue')
axes[0].set_title("Top Features Contributing to PC1")
axes[0].set_xlabel("Loading Value")
axes[0].set_ylabel("Features")

# PC2 plot
sns.barplot(x=pc2_loadings.values[:10], y=pc2_loadings.index[:10], ax=axes[1], color='red')
axes[1].set_title("Top Features Contributing to PC2")
axes[1].set_xlabel("Loading Value")
axes[1].set_ylabel("Features")

plt.tight_layout()
plt.show()


## ctr

In [ ]:
X_c = control.drop(['Asthma', 'donor', 'sex'], axis=1)
X_c = pca.fit_transform(X_c)

pca_columns = [f'PC{i+1}' for i in range(X_c.shape[1])]
X_c_df= pd.DataFrame(X_c, columns=pca_columns)

X_c_df = pd.concat([X_c_df, control[['Asthma', 'donor','sex']]], axis=1)
X_c_df['sex'] = X_c_df['sex'].apply(lambda x: 'Male' if x == 0 else 'Female')

In [ ]:
palette = sns.color_palette('Set1', n_colors=X_c_df['sex'].nunique())
sex_categories = X_c_df['sex'].unique()
color_dict = dict(zip(sex_categories, palette))

# Create scatter plot with mapped colors
scatter = sns.scatterplot(x='PC1', y='PC2', data=X_c_df, hue='sex', palette=color_dict)

# Annotate each point with its 'donor' value
for i, row in X_c_df.iterrows():
    plt.text(row['PC1'], row['PC2'], str(row['donor']), fontsize=8, ha='right', alpha=0.7)

# Compute centroids and plot circles
all_x, all_y = [], []  # Store all points for axis limit adjustment
for sex_group in sex_categories:
    subset = X_c_df[X_c_df['sex'] == sex_group]
    if not subset.empty:
        # Compute centroid
        centroid_x, centroid_y = subset[['PC1', 'PC2']].mean()
        
        # Compute max Euclidean distance for circle radius
        max_distance = np.max(np.sqrt((subset['PC1'] - centroid_x) ** 2 + (subset['PC2'] - centroid_y) ** 2))
        
        # Draw a circle with the same color as the points
        circle = plt.Circle((centroid_x, centroid_y), max_distance, color=color_dict[sex_group], fill=False, linestyle='--', linewidth=2)
        plt.gca().add_patch(circle)

        # Store values to adjust axis limits
        all_x.extend(subset['PC1'])
        all_y.extend(subset['PC2'])
        all_x.append(centroid_x + max_distance)
        all_x.append(centroid_x - max_distance)
        all_y.append(centroid_y + max_distance)
        all_y.append(centroid_y - max_distance)

# Labels and title
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA (ctr)')

# Expand axis limits to fit the circles
plt.xlim(min(all_x) - 2, max(all_x) + 2)
plt.ylim(min(all_y) - 2, max(all_y) + 2)

# Adjust legend positioning
plt.legend(title='Sex', bbox_to_anchor=(1.05, 1), loc='upper left')

# Final layout adjustment
plt.tight_layout()
plt.show()

In [ ]:
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

plt.figure(figsize=(10, 6))
plt.plot(np.arange(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', linestyle='-', color='r', label='Cumulative Explained Variance')
plt.axhline(y=0.9, color='gray', linestyle='--', label='90% Variance Explained')  # 90% threshold line
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("Cumulative Variance Explained by PCA")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
features = control.drop(['Asthma', 'donor', 'sex'], axis=1).columns
pca_loadings = pd.DataFrame(pca.components_, columns=features, index=[f'PC{i+1}' for i in range(pca.n_components_)])
top_5_features = pca_loadings.abs().apply(lambda x: x.nlargest(5).index.tolist(), axis=1)
print(top_5_features)

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(X_c[:, 0], X_c[:, 1], c=control['IL-33'], cmap='viridis')
plt.colorbar(label='IL-33')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA (ctr)')
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(X_c[:, 0], X_c[:, 1], c=control['PDGF-AA'], cmap='viridis')
plt.colorbar(label='PDGF-AA')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA (ctr)')
plt.show()

In [ ]:
# Extract PC1 and PC2 loadings
pc1_loadings = pca_loadings.loc["PC1"].sort_values(key=abs, ascending=False)
pc2_loadings = pca_loadings.loc["PC2"].sort_values(key=abs, ascending=False)

# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# PC1 plot
sns.barplot(x=pc1_loadings.values[:10], y=pc1_loadings.index[:10], ax=axes[0], color='blue')
axes[0].set_title("Top Features Contributing to PC1 (ctr)")
axes[0].set_xlabel("Loading Value")
axes[0].set_ylabel("Features")

# PC2 plot
sns.barplot(x=pc2_loadings.values[:10], y=pc2_loadings.index[:10], ax=axes[1], color='red')
axes[1].set_title("Top Features Contributing to PC2 (ctr)")
axes[1].set_xlabel("Loading Value")
axes[1].set_ylabel("Features")

plt.tight_layout()
plt.show()


## hst

In [ ]:
history = history.reset_index(drop=True)

In [ ]:
X_h = history.drop(['Asthma', 'donor', 'sex'], axis=1)
X_h = pca.fit_transform(X_h)

pca_columns = [f'PC{i+1}' for i in range(X_h.shape[1])]
X_h_df= pd.DataFrame(X_h, columns=pca_columns)

X_h_df = pd.concat([X_h_df, history[['Asthma', 'donor','sex']]], axis=1)

In [ ]:
palette = sns.color_palette('Set1', n_colors=X_h_df['sex'].nunique())
sex_categories = X_h_df['sex'].unique()
color_dict = dict(zip(sex_categories, palette))

# Create scatter plot with mapped colors
scatter = sns.scatterplot(x='PC1', y='PC2', data=X_h_df, hue='sex', palette=color_dict)

# Annotate each point with its 'donor' value
for i, row in X_h_df.iterrows():
    plt.text(row['PC1'], row['PC2'], str(row['donor']), fontsize=8, ha='right', alpha=0.7)

# Compute centroids and plot circles
all_x, all_y = [], []  # Store all points for axis limit adjustment
for sex_group in sex_categories:
    subset = X_h_df[X_h_df['sex'] == sex_group]
    if not subset.empty:
        # Compute centroid
        centroid_x, centroid_y = subset[['PC1', 'PC2']].mean()
        
        # Compute max Euclidean distance for circle radius
        max_distance = np.max(np.sqrt((subset['PC1'] - centroid_x) ** 2 + (subset['PC2'] - centroid_y) ** 2))
        
        # Draw a circle with the same color as the points
        circle = plt.Circle((centroid_x, centroid_y), max_distance, color=color_dict[sex_group], fill=False, linestyle='--', linewidth=2)
        plt.gca().add_patch(circle)

        # Store values to adjust axis limits
        all_x.extend(subset['PC1'])
        all_y.extend(subset['PC2'])
        all_x.append(centroid_x + max_distance)
        all_x.append(centroid_x - max_distance)
        all_y.append(centroid_y + max_distance)
        all_y.append(centroid_y - max_distance)

# Labels and title
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA (hst)')

# Expand axis limits to fit the circles
plt.xlim(min(all_x) - 2, max(all_x) + 2)
plt.ylim(min(all_y) - 2, max(all_y) + 2)

# Adjust legend positioning
plt.legend(title='Sex', bbox_to_anchor=(1.05, 1), loc='upper left')

# Final layout adjustment
plt.tight_layout()
plt.show()

In [ ]:
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

plt.figure(figsize=(10, 6))
plt.plot(np.arange(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', linestyle='-', color='r', label='Cumulative Explained Variance')
plt.axhline(y=0.9, color='gray', linestyle='--', label='90% Variance Explained')  # 90% threshold line
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("Cumulative Variance Explained by PCA")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
features = history.drop(['Asthma', 'donor', 'sex'], axis=1).columns
pca_loadings = pd.DataFrame(pca.components_, columns=features, index=[f'PC{i+1}' for i in range(pca.n_components_)])
top_5_features = pca_loadings.abs().apply(lambda x: x.nlargest(5).index.tolist(), axis=1)
print(top_5_features)

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(X_h[:, 0], X_h[:, 1], c=history['MCP-2'], cmap='viridis')
plt.colorbar(label='MCP-2')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA (hst)')
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(X_h[:, 0], X_h[:, 1], c=history['IL-1b'], cmap='viridis')
plt.colorbar(label='IL-1b')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA (hst)')
plt.show()

In [ ]:
# Extract PC1 and PC2 loadings
pc1_loadings = pca_loadings.loc["PC1"].sort_values(key=abs, ascending=False)
pc2_loadings = pca_loadings.loc["PC2"].sort_values(key=abs, ascending=False)

# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# PC1 plot
sns.barplot(x=pc1_loadings.values[:10], y=pc1_loadings.index[:10], ax=axes[0], color='blue')
axes[0].set_title("Top Features Contributing to PC1 (hst)")
axes[0].set_xlabel("Loading Value")
axes[0].set_ylabel("Features")

# PC2 plot
sns.barplot(x=pc2_loadings.values[:10], y=pc2_loadings.index[:10], ax=axes[1], color='red')
axes[1].set_title("Top Features Contributing to PC2 (hst)")
axes[1].set_xlabel("Loading Value")
axes[1].set_ylabel("Features")

plt.tight_layout()
plt.show()

## fatal

In [ ]:
fatal = fatal.reset_index(drop=True)
fatal

In [ ]:
X_f = fatal.drop(['Asthma', 'donor', 'sex'], axis=1)
X_f = pca.fit_transform(X_f)

pca_columns = [f'PC{i+1}' for i in range(X_f.shape[1])]
X_f_df= pd.DataFrame(X_f, columns=pca_columns)

X_f_df = pd.concat([X_f_df, fatal[['Asthma', 'donor','sex']]], axis=1)

In [ ]:
palette = sns.color_palette('Set1', n_colors=X_f_df['sex'].nunique())
sex_categories = X_f_df['sex'].unique()
color_dict = dict(zip(sex_categories, palette))

# Create scatter plot with mapped colors
scatter = sns.scatterplot(x='PC1', y='PC2', data=X_f_df, hue='sex', palette=color_dict)

# Annotate each point with its 'donor' value
for i, row in X_f_df.iterrows():
    plt.text(row['PC1'], row['PC2'], str(row['donor']), fontsize=8, ha='right', alpha=0.7)

# Compute centroids and plot circles
all_x, all_y = [], []  # Store all points for axis limit adjustment
for sex_group in sex_categories:
    subset = X_f_df[X_f_df['sex'] == sex_group]
    if not subset.empty:
        # Compute centroid
        centroid_x, centroid_y = subset[['PC1', 'PC2']].mean()
        
        # Compute max Euclidean distance for circle radius
        max_distance = np.max(np.sqrt((subset['PC1'] - centroid_x) ** 2 + (subset['PC2'] - centroid_y) ** 2))
        
        # Draw a circle with the same color as the points
        circle = plt.Circle((centroid_x, centroid_y), max_distance, color=color_dict[sex_group], fill=False, linestyle='--', linewidth=2)
        plt.gca().add_patch(circle)

        # Store values to adjust axis limits
        all_x.extend(subset['PC1'])
        all_y.extend(subset['PC2'])
        all_x.append(centroid_x + max_distance)
        all_x.append(centroid_x - max_distance)
        all_y.append(centroid_y + max_distance)
        all_y.append(centroid_y - max_distance)

# Labels and title
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA (ftl)')

# Expand axis limits to fit the circles
plt.xlim(min(all_x) - 2, max(all_x) + 2)
plt.ylim(min(all_y) - 2, max(all_y) + 2)

# Adjust legend positioning
plt.legend(title='Sex', bbox_to_anchor=(1.05, 1), loc='upper left')

# Final layout adjustment
plt.tight_layout()
plt.show()

In [ ]:
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

plt.figure(figsize=(10, 6))
plt.plot(np.arange(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', linestyle='-', color='r', label='Cumulative Explained Variance')
plt.axhline(y=0.9, color='gray', linestyle='--', label='90% Variance Explained')  # 90% threshold line
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("Cumulative Variance Explained by PCA")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
features = fatal.drop(['Asthma', 'donor', 'sex'], axis=1).columns
pca_loadings = pd.DataFrame(pca.components_, columns=features, index=[f'PC{i+1}' for i in range(pca.n_components_)])
top_5_features = pca_loadings.abs().apply(lambda x: x.nlargest(5).index.tolist(), axis=1)
print(top_5_features)

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(X_f[:, 0], X_f[:, 1], c=fatal['IL-28A'], cmap='viridis')
plt.colorbar(label='IL-28A')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA (ftl)')
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(X_f[:, 0], X_f[:, 1], c=fatal['TNFb'], cmap='viridis')
plt.colorbar(label='TNFb')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA (ftl)')
plt.show()

In [ ]:
# Extract PC1 and PC2 loadings
pc1_loadings = pca_loadings.loc["PC1"].sort_values(key=abs, ascending=False)
pc2_loadings = pca_loadings.loc["PC2"].sort_values(key=abs, ascending=False)

# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# PC1 plot
sns.barplot(x=pc1_loadings.values[:10], y=pc1_loadings.index[:10], ax=axes[0], color='blue')
axes[0].set_title("Top Features Contributing to PC1 (ftl)")
axes[0].set_xlabel("Loading Value")
axes[0].set_ylabel("Features")

# PC2 plot
sns.barplot(x=pc2_loadings.values[:10], y=pc2_loadings.index[:10], ax=axes[1], color='red')
axes[1].set_title("Top Features Contributing to PC2 (ftl)")
axes[1].set_xlabel("Loading Value")
axes[1].set_ylabel("Features")

plt.tight_layout()
plt.show()

## Model

In [ ]:
df = df.drop('donor', axis = 1)
X = df.drop('Asthma', axis = 1)
X

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, df['Asthma'], test_size=0.2, random_state=0)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import cross_val_score

model = RandomForestClassifier(n_estimators=100, random_state=0)
model.fit(X_train, y_train)

scores = cross_val_score(model, X, df['Asthma'], cv=5)
print("Cross-validation scores:", scores)
print("Mean cross-validation score:", scores.mean())

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

In [ ]:
importances = model.feature_importances_
importances_df = pd.DataFrame({
    'feature': X.columns,
    'importance': importances
})
importances_df.sort_values(by='importance', ascending=False).head(10)


In [ ]:
confusion_matrix(y_test, y_pred)
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='YlGnBu')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
from sklearn.svm import SVC

model_svm = SVC(kernel='linear')
model_svm.fit(X_train, y_train)

scores_svm = cross_val_score(model_svm, X, df['Asthma'], cv=5)
print("Cross-validation scores:", scores_svm)
print("Mean cross-validation score:", scores_svm.mean())

y_pred_svm = model_svm.predict(X_test)
accuracy_svm = accuracy_score(y_test, y_pred_svm)
print("Accuracy:", accuracy_svm)

In [ ]:
confusion_matrix_svm = confusion_matrix(y_test, y_pred_svm)
sns.heatmap(confusion_matrix_svm, annot=True, fmt='d', cmap='YlGnBu')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
#what about logistic regression
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression()
model_lr.fit(X_train, y_train)

scores_lr = cross_val_score(model_lr, X, df['Asthma'], cv=5)
print("Cross-validation scores:", scores_lr)
print("Mean cross-validation score:", scores_lr.mean())

y_pred_lr = model_lr.predict(X_test)
accuracy_lr = accuracy_score(y_test, y_pred_lr)
print("Accuracy:", accuracy_lr)

In [ ]:
importances_lr = model_lr.coef_[0]
importances_lr_df = pd.DataFrame({
    'feature': X.columns,
    'importance': importances_lr
})
importances_lr_df.sort_values(by='importance', ascending=False).head(10)

In [ ]:
confusion_matrix_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(confusion_matrix_lr, annot=True, fmt='d', cmap='YlGnBu')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## clusters

In [ ]:
import scanpy as sc
df = pd.read_excel('stat test_01APR25.xlsx')
df['sex'] = df['sex'].apply(lambda x: 0 if x == 'MALE' else 1)
#if Asthma == Control, set 0, if Asthma == History, set 1, if Asthma == fatal, set 2
df['Asthma'] = df['Asthma'].apply(lambda x: 2 if x == 'Fatal' else (1 if x == 'History' else 0))
df

In [ ]:
df_numeric = df.select_dtypes(include=[np.number])
list_metadata = {'donor' : df.donor, 'Asthma' : df.Asthma, 'sex': df.sex}
df_metadata = pd.DataFrame(list_metadata)

In [ ]:
adata = sc.AnnData(df_numeric)
df_metadata.index = adata.obs.index
adata.obs['donor'] = df_metadata.donor
adata.obs['Asthma'] = df_metadata.Asthma
adata.obs['sex'] = df_metadata.sex

In [ ]:
sc.pp.scale(adata, max_value=5)

In [ ]:
sc.tl.pca(adata, svd_solver="arpack")
sc.pl.pca_variance_ratio(adata, log=False)

In [ ]:
sc.pl.pca_loadings(adata, components = '1,2')

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, n_pcs= 15, random_state=0)
sc.tl.leiden(adata, resolution=0.8, flavor='leidenalg')
sc.tl.paga(adata)
sc.pl.paga(adata)
sc.tl.umap(adata, init_pos='paga', random_state=0)

In [ ]:
markers = list(adata.var_names)

In [ ]:
adata.obs.rename(columns={"Asthma": "Asthma_status"}, inplace=True)
sc.pl.umap(adata, color="Asthma_status", cmap='turbo', size=300)


In [ ]:
adata.obs.rename(columns={"Asthma": "Asthma_status"}, inplace=True)
sc.pl.umap(adata, color= ['leiden'], cmap='turbo', size = 300)

In [ ]:
adata.obs

In [ ]:
adata.obs.rename(columns={"sex": "sex_status"}, inplace=True)

sc.pl.umap(adata, color=['leiden', 'BMI', 'sex_status', 'Asthma_status'], size = 300)

In [ ]:
sc.tl.rank_genes_groups(adata, "leiden", method="wilcoxon")
df = pd.DataFrame(
    {group: adata.uns["rank_genes_groups"]["names"][group] for group in adata.uns["rank_genes_groups"]["names"].dtype.names}
)
df.head(3)

In [ ]:
result = adata.uns["rank_genes_groups"]
groups = result["names"].dtype.names

celltype = {'celltype': []}
cluster_to_genes = {}
for group in groups:
    top_genes = result["names"][group][:3]
    cluster_to_genes[group] = f"{':'.join(top_genes)} ({group})"

celltype['celltype'] = [cluster_to_genes[leiden] for leiden in adata.obs['leiden']]
adata.obs["celltype"] = celltype['celltype']

print(adata.obs[["leiden", "celltype"]].head())

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

sc.pl.umap(
    adata,
    color=['celltype'],
    cmap='turbo',
    title = 'celltypes',
    show=False,  
    size = 300 
)

ax = plt.gca() 
for cluster in adata.obs['leiden'].cat.categories:
    cluster_mask = adata.obs['leiden'] == cluster
    cluster_coords = adata.obsm['X_umap'][cluster_mask]
    x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()
    ax.text(x, y, cluster, color='black', fontsize=10, weight='bold', ha='center', va='center')

plt.show()

In [ ]:
cluster_1 = adata[adata.obs['leiden'] == '1']
cluster_1.obs

In [ ]:
# Set style
sns.set_style("whitegrid")

# Create subplots for better visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot sex distribution
sns.countplot(x="sex_status", data=cluster_1.obs, ax=axes[0], palette="coolwarm")
axes[0].set_title("Sex Distribution")
axes[0].set_xlabel("Sex Status")
axes[0].set_ylabel("Count")

# Plot Asthma distribution
sns.countplot(x="Asthma_status", data=cluster_1.obs, ax=axes[1], palette="viridis")
axes[1].set_title("Asthma Status Distribution")
axes[1].set_xlabel("Asthma Status")
axes[1].set_ylabel("Count")

# Show the plots
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# Ensure 'donor' is a categorical variable
cluster_1.obs['donor'] = cluster_1.obs['donor'].astype("category")

# Now plot the UMAP
sc.pl.umap(cluster_1, color=['donor'], palette='tab20', size=300)


## unsupervised

In [ ]:
df = pd.read_excel('stat test_01APR25.xlsx')
df['sex'] = df['sex'].apply(lambda x: 0 if x == 'MALE' else 1)
df['Asthma'] = df['Asthma'].apply(lambda x: 2 if x == 'Fatal' else (1 if x == 'History' else 0))
#only include the columns age, asthma, bmi, donor, and sex
df_demographic = df[['age', 'BMI', 'Asthma', 'sex']]

In [ ]:
#make unsupervised clusters of rows in df_demographic
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=0).fit(df_demographic)
df_demographic['cluster'] = kmeans.labels_

In [ ]:
df_demographic

In [ ]:
#sort by age
df_demographic = df_demographic.sort_values(by='age', ascending=True)
df_demographic

In [ ]:
#visualize clusters
sns.pairplot(df_demographic, hue='cluster')

In [ ]:
df_demographic['donor'] = df.donor
df_demographic

## asthmatics

In [ ]:
df = pd.read_excel('stat test_01APR25.xlsx')
df['sex'] = df['sex'].apply(lambda x: 0 if x == 'MALE' else 1)

In [ ]:
df_asthmatics = df[(df['age'] >= 40)]
df_asthmatics.head()

In [ ]:
df_asthmatics = df_asthmatics.reset_index(drop=True)
df_asthmatics.head()

In [ ]:
exclude_cols = ['Asthma', 'donor', 'sex']
df_asthmatics.loc[:, ~df_asthmatics.columns.isin(exclude_cols)] = scaler.fit_transform(df_asthmatics.loc[:, ~df_asthmatics.columns.isin(exclude_cols)])

In [ ]:
pca = PCA(n_components=0.9)

X = df_asthmatics.drop(['Asthma', 'donor', 'sex'], axis=1)
X = pca.fit_transform(X)

pca_columns = [f'PC{i+1}' for i in range(X.shape[1])]
X_df= pd.DataFrame(X, columns=pca_columns)

X_df = pd.concat([X_df, df_asthmatics[['Asthma', 'donor','sex']]], axis=1)
X_df['sex'] = X_df['sex'].apply(lambda x: 'Male' if x == 0 else 'Female')

In [ ]:
X_df.head()

In [ ]:
palette = sns.color_palette('Set1', n_colors=X_df['Asthma'].nunique())
sex_categories = X_df['Asthma'].unique()
color_dict = dict(zip(sex_categories, palette))

# Create scatter plot with mapped colors
scatter = sns.scatterplot(x='PC1', y='PC2', data=X_df, hue='Asthma', palette=color_dict)

# Annotate each point with its 'donor' value
for i, row in X_df.iterrows():
    plt.text(row['PC1'], row['PC2'], str(row['donor']), fontsize=8, ha='right', alpha=0.7)

# Compute centroids and plot circles
all_x, all_y = [], []  # Store all points for axis limit adjustment
for sex_group in sex_categories:
    subset = X_df[X_df['Asthma'] == sex_group]
    if not subset.empty:
        # Compute centroid
        centroid_x, centroid_y = subset[['PC1', 'PC2']].mean()
        
        # Compute max Euclidean distance for circle radius
        max_distance = np.max(np.sqrt((subset['PC1'] - centroid_x) ** 2 + (subset['PC2'] - centroid_y) ** 2))
        
        # Draw a circle with the same color as the points
        circle = plt.Circle((centroid_x, centroid_y), max_distance, color=color_dict[sex_group], fill=False, linestyle='--', linewidth=2)
        plt.gca().add_patch(circle)

        # Store values to adjust axis limits
        all_x.extend(subset['PC1'])
        all_y.extend(subset['PC2'])
        all_x.append(centroid_x + max_distance)
        all_x.append(centroid_x - max_distance)
        all_y.append(centroid_y + max_distance)
        all_y.append(centroid_y - max_distance)

# Labels and title
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA')

# Expand axis limits to fit the circles
plt.xlim(min(all_x) - 2, max(all_x) + 2)
plt.ylim(min(all_y) - 2, max(all_y) + 2)

# Adjust legend positioning
plt.legend(title='Asthma', bbox_to_anchor=(1.05, 1), loc='upper left')

# Final layout adjustment
plt.tight_layout()
plt.show()

In [ ]:
palette = sns.color_palette('Set1', n_colors=X_df['sex'].nunique())
sex_categories = X_df['sex'].unique()
color_dict = dict(zip(sex_categories, palette))

# Create scatter plot with mapped colors
scatter = sns.scatterplot(x='PC1', y='PC2', data=X_df, hue='sex', palette=color_dict)

# Annotate each point with its 'donor' value
for i, row in X_df.iterrows():
    plt.text(row['PC1'], row['PC2'], str(row['donor']), fontsize=8, ha='right', alpha=0.7)

# Compute centroids and plot circles
all_x, all_y = [], []  # Store all points for axis limit adjustment
for sex_group in sex_categories:
    subset = X_df[X_df['sex'] == sex_group]
    if not subset.empty:
        # Compute centroid
        centroid_x, centroid_y = subset[['PC1', 'PC2']].mean()
        
        # Compute max Euclidean distance for circle radius
        max_distance = np.max(np.sqrt((subset['PC1'] - centroid_x) ** 2 + (subset['PC2'] - centroid_y) ** 2))
        
        # Draw a circle with the same color as the points
        circle = plt.Circle((centroid_x, centroid_y), max_distance, color=color_dict[sex_group], fill=False, linestyle='--', linewidth=2)
        plt.gca().add_patch(circle)

        # Store values to adjust axis limits
        all_x.extend(subset['PC1'])
        all_y.extend(subset['PC2'])
        all_x.append(centroid_x + max_distance)
        all_x.append(centroid_x - max_distance)
        all_y.append(centroid_y + max_distance)
        all_y.append(centroid_y - max_distance)

# Labels and title
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA')

# Expand axis limits to fit the circles
plt.xlim(min(all_x) - 2, max(all_x) + 2)
plt.ylim(min(all_y) - 2, max(all_y) + 2)

# Adjust legend positioning
plt.legend(title='sex', bbox_to_anchor=(1.05, 1), loc='upper left')

# Final layout adjustment
plt.tight_layout()
plt.show()

In [ ]:
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

plt.figure(figsize=(10, 6))
plt.plot(np.arange(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', linestyle='-', color='r', label='Cumulative Explained Variance')
plt.axhline(y=0.9, color='gray', linestyle='--', label='90% Variance Explained')  # 90% threshold line
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("Cumulative Variance Explained by PCA")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
features = df_asthmatics.drop(['Asthma', 'donor', 'sex'], axis=1).columns
pca_loadings = pd.DataFrame(pca.components_, columns=features, index=[f'PC{i+1}' for i in range(pca.n_components_)])
top_5_features = pca_loadings.abs().apply(lambda x: x.nlargest(5).index.tolist(), axis=1)
print(top_5_features)

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(X[:, 0], X[:, 1], c=df_asthmatics['IL-9'], cmap='viridis')
plt.colorbar(label='IL-9')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA')
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(X[:, 0], X[:, 1], c=df_asthmatics['M-CSF'], cmap='viridis')
plt.colorbar(label='M-CSF')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA')
plt.show()

In [ ]:
# Extract PC1 and PC2 loadings
pc1_loadings = pca_loadings.loc["PC1"].sort_values(key=abs, ascending=False)
pc2_loadings = pca_loadings.loc["PC2"].sort_values(key=abs, ascending=False)

# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# PC1 plot
sns.barplot(x=pc1_loadings.values[:10], y=pc1_loadings.index[:10], ax=axes[0], color='blue')
axes[0].set_title("Top Features Contributing to PC1")
axes[0].set_xlabel("Loading Value")
axes[0].set_ylabel("Features")

# PC2 plot
sns.barplot(x=pc2_loadings.values[:10], y=pc2_loadings.index[:10], ax=axes[1], color='red')
axes[1].set_title("Top Features Contributing to PC2")
axes[1].set_xlabel("Loading Value")
axes[1].set_ylabel("Features")

plt.tight_layout()
plt.show()

In [ ]:
df_asthmatics = df_asthmatics.drop('donor', axis = 1)
X = df_asthmatics.drop('Asthma', axis = 1)
X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, df_asthmatics['Asthma'], test_size=0.25, random_state=0)

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=0)
model.fit(X_train, y_train)

scores = cross_val_score(model, X, df_asthmatics['Asthma'], cv=5)
print("Cross-validation scores:", scores)
print("Mean cross-validation score:", scores.mean())

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

In [ ]:
importances = model.feature_importances_
importances_df = pd.DataFrame({
    'feature': X.columns,
    'importance': importances
})
importances_df.sort_values(by='importance', ascending=False).head(10)

In [ ]:
y_test

In [ ]:
class_names = ['Control', 'Fatal', 'History'] 

# Compute the confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot the confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu')

# Set axis labels and class names
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.xticks(ticks=[0.5, 1.5, 2.5], labels=class_names)
plt.yticks(ticks=[0.5, 1.5, 2.5], labels=class_names, rotation=0)

# Show the plot
plt.show()


In [ ]:
df = pd.read_excel('stat test_01APR25.xlsx')
df['sex'] = df['sex'].apply(lambda x: 0 if x == 'MALE' else 1)
#if Asthma == Control, set 0, if Asthma == History, set 1, if Asthma == fatal, set 2
df['Asthma'] = df['Asthma'].apply(lambda x: 2 if x == 'Fatal' else (1 if x == 'History' else 0))
df_asthmatics = df[(df['BMI'] >= 30)]

In [ ]:
df_numeric = df_asthmatics.select_dtypes(include=[np.number])
list_metadata = {'donor' : df_asthmatics.donor, 'Asthma' : df_asthmatics.Asthma, 'sex': df_asthmatics.sex}
df_metadata = pd.DataFrame(list_metadata)

In [ ]:
adata = sc.AnnData(df_numeric)
df_metadata.index = adata.obs.index
adata.obs['donor'] = df_metadata.donor
adata.obs['Asthma'] = df_metadata.Asthma
adata.obs['sex'] = df_metadata.sex

In [ ]:
sc.pp.scale(adata, max_value=5)

In [ ]:
sc.tl.pca(adata, svd_solver="arpack")
sc.pl.pca_variance_ratio(adata, log=False)

In [ ]:
sc.pl.pca_loadings(adata, components = '1,2')

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, n_pcs= 15, random_state=0)
sc.tl.leiden(adata, resolution=0.8, flavor='leidenalg')
sc.tl.paga(adata)
sc.pl.paga(adata)
sc.tl.umap(adata, init_pos='paga', random_state=0)

In [ ]:
adata.obs.rename(columns={"Asthma": "Asthma_status"}, inplace=True)
sc.pl.umap(adata, color="Asthma_status", cmap='turbo', size=300)

In [ ]:
sc.pl.umap(adata, color= ['leiden'], cmap='turbo', size = 300)

In [ ]:
adata.obs.rename(columns={"sex": "sex_status"}, inplace=True)

sc.pl.umap(adata, color=['leiden', 'BMI', 'sex_status', 'Asthma_status'], size = 300)

In [ ]:
sc.tl.rank_genes_groups(adata, "leiden", method="wilcoxon")
df = pd.DataFrame(
    {group: adata.uns["rank_genes_groups"]["names"][group] for group in adata.uns["rank_genes_groups"]["names"].dtype.names}
)
df.head(3)

In [ ]:
result = adata.uns["rank_genes_groups"]
groups = result["names"].dtype.names

celltype = {'celltype': []}
cluster_to_genes = {}
for group in groups:
    top_genes = result["names"][group][:3]
    cluster_to_genes[group] = f"{':'.join(top_genes)} ({group})"

celltype['celltype'] = [cluster_to_genes[leiden] for leiden in adata.obs['leiden']]
adata.obs["celltype"] = celltype['celltype']

In [ ]:
sc.pl.umap(
    adata,
    color=['celltype'],
    cmap='turbo',
    title = 'celltypes',
    show=False,  
    size = 300 
)

ax = plt.gca() 
for cluster in adata.obs['leiden'].cat.categories:
    cluster_mask = adata.obs['leiden'] == cluster
    cluster_coords = adata.obsm['X_umap'][cluster_mask]
    x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()
    ax.text(x, y, cluster, color='black', fontsize=10, weight='bold', ha='center', va='center')

plt.show()

In [ ]:
cluster_1 = adata[adata.obs['leiden'] == '0']
cluster_1.obs

In [ ]:
donors = cluster_1.obs['donor'].unique()  # get unique donor names
donors_1 = df_asthmatics[df_asthmatics['donor'].isin(donors)]


In [ ]:
# Set style
sns.set_style("whitegrid")

# Create subplots for better visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))


sns.countplot(x="sex_status", data=cluster_1.obs, ax=axes[0], palette="coolwarm")
axes[0].set_title("Sex Distribution")
axes[0].set_xlabel("Sex Status")
axes[0].set_ylabel("Count")

# #plot age distribution
# sns.histplot(donors_1['age'], bins=15, ax=axes[0], color="skyblue")
# axes[0].set_title("Age Distribution")
# axes[0].set_xlabel("Age")


# Plot Asthma distribution
sns.countplot(x="Asthma_status", data=cluster_1.obs, ax=axes[1], palette="viridis")
axes[1].set_title("Asthma Status Distribution")
axes[1].set_xlabel("Asthma Status")
axes[1].set_ylabel("Count")

# Show the plots
plt.tight_layout()
plt.show()

In [ ]:
# Ensure 'donor' is a categorical variable
cluster_1.obs['donor'] = cluster_1.obs['donor'].astype("category")

# Now plot the UMAP
sc.pl.umap(cluster_1, color=['donor'], palette='tab20', size=300)


In [ ]:
df = pd.read_excel('stat test_01APR25.xlsx')
df['sex'] = df['sex'].apply(lambda x: 0 if x == 'MALE' else 1)
df['Asthma'] = df['Asthma'].apply(lambda x: 2 if x == 'Fatal' else (1 if x == 'History' else 0))
#only include the columns age, asthma, bmi, donor, and sex
df_demographic = df[['age', 'BMI', 'Asthma', 'sex']]
#select asthmatics
df_asthmatics = df_demographic[(df_demographic['BMI'] >= 30)]

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=0).fit(df_asthmatics)
df_asthmatics['cluster'] = kmeans.labels_

In [ ]:
df_asthmatics

In [ ]:
sns.pairplot(df_asthmatics, hue='cluster')